In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

from core.logging import configure_logging
from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from registry import RegistryClient

from gl.models import GLSegments
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from break_analysis.tools import RegistryTools
from break_analysis import BreakAnalysisAgent
from break_analysis.models import BreakRecord
from break_analysis.builder import BreakCaseBuilder


configure_logging()

def display_df(df):
    display(df.toPandas())

In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('break-agent-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/12 13:24:54 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/09/12 13:24:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-60ecc554-fe69-46b7-9c50-e3eb63bb1591;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 71ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#

In [4]:
registry_client = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)

gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry_client)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

registry_tools = RegistryTools(registry_client=registry_client)

In [5]:
llm = ChatOllama(
    model='qwen3:14b-q4_K_M',
    temperature=0
)

agent = BreakAnalysisAgent(
    llm=llm,
    registry_tools=registry_tools
)

In [6]:
workflow_run_id = '694af9a4-9c03-419f-a9d6-705ab094677d'

recon_df = recon.get_results(workflow_run_id)
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

breaks = []
for row in breaks_df.collect():
    segments = GLSegments(
        entity_cd = row['ENTITY_CD'],
        branch_cd = row['BRANCH_CD'],
        dept_cd = row['DEPT_CD'],
        gl_account = row['GL_ACCOUNT'],
        sub_account = row['SUB_ACCOUNT'],
        affiliate_cd = row['AFFILIATE_CD'],
        product_cd = row['PRODUCT_CD'],
        book_cd = row['BOOK_CD'],
        source_cd = row['SOURCE_CD'],
    )
    breaks.append(
        BreakRecord(
            recon_result_id= row['RECON_RESULT_ID'],
            workflow_run_id = row['WORKFLOW_RUN_ID'],
            as_of_date = row['AS_OF_DATE'],
            segments = segments,
            accounted_currency = row['ACCOUNTED_CURRENCY'],
            interface_balance = row['INTERFACE_BALANCE'],
            gl_balance = row['GL_BALANCE'],
            difference_amount = row['DIFFERENCE_AMOUNT']
        )
    )

segment_defaults = gl.get_segment_defaults()
case_builder = BreakCaseBuilder(segment_defaults)

break_cases = case_builder.build(breaks)

2026-09-12 13:24:58,822 | INFO | break_analysis.builder | Building break cases | records=7
2026-09-12 13:24:58,823 | INFO | break_analysis.builder | Break cases built | total=3 | one_to_one=2 | many_to_one=1 | ambiguous=0 | interface_only=0 | gl_only=0 | unmatched=0 | duration_ms=1
7 3


In [7]:
break_case = break_cases[0]

agent.analyze(break_case)

2026-09-12 13:24:58,827 | INFO | break_analysis.agent | Analyzing break case | case_id=b2956924 | workflow_run_id=694af9a4 | topology=MANY_TO_ONE | records=3
2026-09-12 13:24:58,828 | INFO | break_analysis.agent | Invoking LLM | case_id=b2956924 | round=1
2026-09-12 13:25:11,333 | INFO | break_analysis.agent | LLM invoked | case_id=b2956924 | round=1 | tool_calls=4 | input_tokens=1108 | output_tokens=203 | total_tokens=1311 | prompt_eval_count=1108 | eval_count=203 | load_ms=5057 | prompt_eval_ms=818 | eval_ms=6612 | total_ms=12493 | duration_ms=12505
2026-09-12 13:25:11,334 | INFO | break_analysis.agent | Tool round | case_id=b2956924 | round=1 | tool_calls=4
2026-09-12 13:25:11,554 | INFO | break_analysis.agent | Tool invoked | case_id=b2956924 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "310000", "business_dt": "2026-03-31"} | duration_ms=220
2026-09-12 13:25:11,678 | INFO | break_analysis.agent | Tool invoked | case_id=b2956924 | tool=validate_seg

BreakAnalysisResult(case_id=UUID('b2956924-48d2-409c-8df1-991629246375'), recon_result_ids=(UUID('dbc5342b-1bc1-4ef9-a77c-2f8a358d383e'), UUID('4fe199b3-8537-4650-9860-a969989d8d43'), UUID('bd5598a3-6a7c-40d0-a3bc-42d2ca52f930')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation="Invalid segment values found in investigation records: gl_account '310000' (inactive), sub_account '003000' (inactive), and gl_account '4100000' (missing from Registry).")

In [8]:
break_case = break_cases[1]

agent.analyze(break_case)

2026-09-12 13:26:18,294 | INFO | break_analysis.agent | Analyzing break case | case_id=eb71efe8 | workflow_run_id=694af9a4 | topology=ONE_TO_ONE | records=2
2026-09-12 13:26:18,295 | INFO | break_analysis.agent | Invoking LLM | case_id=eb71efe8 | round=1
2026-09-12 13:26:21,019 | INFO | break_analysis.agent | LLM invoked | case_id=eb71efe8 | round=1 | tool_calls=1 | input_tokens=846 | output_tokens=50 | total_tokens=896 | prompt_eval_count=846 | eval_count=50 | load_ms=128 | prompt_eval_ms=324 | eval_ms=1978 | total_ms=2723 | duration_ms=2724
2026-09-12 13:26:21,019 | INFO | break_analysis.agent | Tool round | case_id=eb71efe8 | round=1 | tool_calls=1
2026-09-12 13:26:21,109 | INFO | break_analysis.agent | Tool invoked | case_id=eb71efe8 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "210000", "business_dt": "2026-03-31"} | duration_ms=90
2026-09-12 13:26:21,110 | INFO | break_analysis.agent | Invoking LLM | case_id=eb71efe8 | round=2
2026-09-12 13:26:23

BreakAnalysisResult(case_id=UUID('eb71efe8-8146-41bd-bd4b-3245d49e6ba9'), recon_result_ids=(UUID('e0161602-3a56-4b0c-9165-2a3bfb06f733'), UUID('56113b7e-582f-41d9-98ed-8575fceda4ab')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation="The 'GL_ACCOUNT' segment with value '210000' exists in the Registry but is marked as inactive (status='I') for the business date 2026-03-31.")

In [9]:
break_case = break_cases[2]

agent.analyze(break_case)

2026-09-12 13:26:37,585 | INFO | break_analysis.agent | Analyzing break case | case_id=50c8e6eb | workflow_run_id=694af9a4 | topology=ONE_TO_ONE | records=2
2026-09-12 13:26:37,585 | INFO | break_analysis.agent | Invoking LLM | case_id=50c8e6eb | round=1
2026-09-12 13:26:46,227 | INFO | break_analysis.agent | LLM invoked | case_id=50c8e6eb | round=1 | tool_calls=3 | input_tokens=849 | output_tokens=130 | total_tokens=979 | prompt_eval_count=849 | eval_count=130 | load_ms=140 | prompt_eval_ms=426 | eval_ms=7945 | total_ms=8640 | duration_ms=8641
2026-09-12 13:26:46,228 | INFO | break_analysis.agent | Tool round | case_id=50c8e6eb | round=1 | tool_calls=3
2026-09-12 13:26:46,318 | INFO | break_analysis.agent | Tool invoked | case_id=50c8e6eb | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "", "business_dt": "2026-03-31"} | duration_ms=90
2026-09-12 13:26:46,407 | INFO | break_analysis.agent | Tool invoked | case_id=50c8e6eb | tool=validate_segment | args={"

BreakAnalysisResult(case_id=UUID('50c8e6eb-8418-49db-b016-54294367429f'), recon_result_ids=(UUID('6a1980e4-59fd-4d6b-a905-b745b80468f4'), UUID('0a6a3f0c-b9cf-49ba-8875-14bd28cd23af')), status=<BreakAnalysisStatus.UNEXPLAINED: 'UNEXPLAINED'>, root_cause=None, explanation='The investigation record contains blank values for gl_account, sub_account, and product_cd. Blank values are explicitly excluded from REGISTRY_INVALID_SEGMENT classification per the hypothesis criteria. The pivot record contains valid nonblank values for these segments.')

In [10]:
# spark.stop()